# Heart Disease Prediction Analysis
## CodeAlpha Machine Learning Internship Project

This notebook demonstrates the complete machine learning pipeline for heart disease prediction.

**Author:** CodeAlpha Machine Learning Intern
**Project:** Disease Prediction from Medical Data

## Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load and Inspect Data](#2-load-and-inspect-data)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Feature Analysis](#5-feature-analysis)
6. [Model Training](#6-model-training)
7. [Model Evaluation](#7-model-evaluation)
8. [Model Comparison](#8-model-comparison)
9. [Best Model Selection](#9-best-model-selection)
10. [Predictions](#10-predictions)
11. [Conclusions](#11-conclusions)

## 1. Import Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

# XGBoost
from xgboost import XGBClassifier

# Model persistence
import joblib

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

## 2. Load and Inspect Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/heart.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")  # Excluding target

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary
df.describe().T

In [ ]:
# Check column names
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

### Feature Description

| Column | Description |
|--------|-------------|
| age | Age in years |
| sex | Gender (1 = Male, 0 = Female) |
| cp | Chest Pain Type (0-3) |
| trestbps | Resting Blood Pressure (mm Hg) |
| chol | Serum Cholesterol (mg/dl) |
| fbs | Fasting Blood Sugar > 120 mg/dl (1 = True) |
| restecg | Resting ECG Results (0-2) |
| thalach | Maximum Heart Rate Achieved |
| exang | Exercise Induced Angina (1 = Yes) |
| oldpeak | ST Depression Induced by Exercise |
| slope | Slope of Peak Exercise ST Segment |
| ca | Number of Major Vessels (0-4) |
| thal | Thalassemia (1 = Normal, 2 = Fixed Defect, 3 = Reversible Defect) |
| target | Target Variable (1 = Disease, 0 = No Disease) |

## 3. Exploratory Data Analysis (EDA)

### 3.1 Check for Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values)
print(f"\nTotal Missing Values: {missing_values.sum()}")

### 3.2 Check for Duplicates

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    print(f"\nRemoving {duplicates} duplicates...")
    df = df.drop_duplicates()
    print(f"New dataset shape: {df.shape}")

### 3.3 Target Variable Distribution

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71', '#e74c3c']
target_counts = df['target'].value_counts()

bars = ax.bar(['No Disease (0)', 'Disease (1)'], 
             [target_counts[0], target_counts[1]],
             color=colors, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Target Class', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
ax.set_title('Target Class Distribution', fontsize=16, fontweight='bold')

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height}\n({height/len(df)*100:.1f}%)',
               xy=(bar.get_x() + bar.get_width() / 2, height),
               ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('../reports/plots/target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Target Distribution:")
print(f"  No Disease (0): {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"  Disease (1): {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)")

### 3.4 Correlation Heatmap

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 12))

corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlBu_r',
            center=0, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)

ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/plots/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.5 Age Distribution

In [ ]:
# Age distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['age'], bins=20, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age (years)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Age Distribution (Histogram)', fontsize=14, fontweight='bold')
axes[0].axvline(df['age'].mean(), color='red', linestyle='--', label=f'Mean: {df["age"].mean():.1f}')
axes[0].legend()

# Box plot
sns.boxplot(data=df, y='age', ax=axes[1], color='#3498db')
axes[1].set_ylabel('Age (years)', fontsize=12)
axes[1].set_title('Age Distribution (Box Plot)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/plots/age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Age Statistics:")
print(f"  Mean: {df['age'].mean():.2f} years")
print(f"  Median: {df['age'].median():.2f} years")
print(f"  Std Dev: {df['age'].std():.2f} years")
print(f"  Range: {df['age'].min()} - {df['age'].max()} years")

### 3.6 Cholesterol Distribution

In [ ]:
# Cholesterol distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['chol'], bins=20, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Cholesterol (mg/dl)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Cholesterol Distribution (Histogram)', fontsize=14, fontweight='bold')
axes[0].axvline(df['chol'].mean(), color='red', linestyle='--', label=f'Mean: {df["chol"].mean():.1f}')
axes[0].legend()

# Box plot
sns.boxplot(data=df, y='chol', ax=axes[1], color='#9b59b6')
axes[1].set_ylabel('Cholesterol (mg/dl)', fontsize=12)
axes[1].set_title('Cholesterol Distribution (Box Plot)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/plots/cholesterol_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Cholesterol Statistics:")
print(f"  Mean: {df['chol'].mean():.2f} mg/dl")
print(f"  Median: {df['chol'].median():.2f} mg/dl")
print(f"  Std Dev: {df['chol'].std():.2f} mg/dl")

### 3.7 Heart Disease by Gender

In [ ]:
# Heart disease by gender
fig, ax = plt.subplots(figsize=(10, 6))

gender_map = {0: 'Female', 1: 'Male'}
target_map = {0: 'No Disease', 1: 'Disease'}

df_temp = df.copy()
df_temp['sex_label'] = df_temp['sex'].map(gender_map)
df_temp['target_label'] = df_temp['target'].map(target_map)

colors = ['#2ecc71', '#e74c3c']
sns.countplot(data=df_temp, x='sex_label', hue='target_label', palette=colors, ax=ax)

ax.set_xlabel('Gender', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Heart Disease Distribution by Gender', fontsize=14, fontweight='bold')
ax.legend(title='Diagnosis')

plt.tight_layout()
plt.savefig('../reports/plots/disease_by_gender.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.8 Heart Disease by Chest Pain Type

In [ ]:
# Heart disease by chest pain type
fig, ax = plt.subplots(figsize=(12, 6))

cp_map = {0: 'Typical Angina', 1: 'Atypical Angina', 2: 'Non-anginal Pain', 3: 'Asymptomatic'}

df_temp = df.copy()
df_temp['cp_label'] = df_temp['cp'].map(cp_map)
df_temp['target_label'] = df_temp['target'].map(target_map)

colors = ['#2ecc71', '#e74c3c']
sns.countplot(data=df_temp, x='cp_label', hue='target_label', palette=colors, ax=ax)

ax.set_xlabel('Chest Pain Type', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Heart Disease Distribution by Chest Pain Type', fontsize=14, fontweight='bold')
ax.legend(title='Diagnosis')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../reports/plots/disease_by_chest_pain.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Separate features and target
feature_columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                   'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

X = df[feature_columns]
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"\nTest target distribution:")
print(y_test.value_counts())

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns)

print("Features scaled successfully!")
print(f"\nScaled training data mean (should be ~0):\n{X_train_scaled.mean()}")
print(f"\nScaled training data std (should be ~1):\n{X_train_scaled.std()}")

## 5. Feature Analysis

In [ ]:
# Correlation with target
target_corr = df.corr()['target'].drop('target').abs().sort_values(ascending=False)

print("Feature Correlation with Target (absolute value, sorted):")
print("-" * 50)
for feature, corr in target_corr.items():
    print(f"{feature:12s}: {corr:.4f}")

In [ ]:
# Feature correlation bar plot
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn(np.linspace(0, 1, len(target_corr)))[::-1]
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')

ax.set_xlabel('Absolute Correlation with Target', fontsize=12)
ax.set_title('Feature Importance by Correlation with Target', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Model Training

### 6.1 Initialize Models

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

print("Models initialized:")
for name in models.keys():
    print(f"  - {name}")

### 6.2 Train Models with Default Parameters

In [ ]:
# Train all models and evaluate using cross-validation
cv_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_results[name] = {
        'cv_scores': cv_scores,
        'mean_cv': cv_scores.mean(),
        'std_cv': cv_scores.std()
    }
    
    print(f"  CV Scores: {cv_scores}")
    print(f"  Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\n" + "="*60)
print("Cross-Validation Summary:")
print("="*60)
for name, results in cv_results.items():
    print(f"{name:25s}: {results['mean_cv']:.4f} (+/- {results['std_cv']:.4f})")

### 6.3 Hyperparameter Tuning - Random Forest

In [ ]:
# Hyperparameter tuning for Random Forest
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Tuning Random Forest...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {rf_grid.best_params_}")
print(f"Best CV Score: {rf_grid.best_score_:.4f}")

### 6.4 Hyperparameter Tuning - XGBoost

In [ ]:
# Hyperparameter tuning for XGBoost
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

print("Tuning XGBoost...")
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    xgb_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {xgb_grid.best_params_}")
print(f"Best CV Score: {xgb_grid.best_score_:.4f}")

In [ ]:
# Store best models
best_models = {
    'Logistic Regression': models['Logistic Regression'],
    'Random Forest': rf_grid.best_estimator_,
    'XGBoost': xgb_grid.best_estimator_
}

print("Best models after tuning:")
for name, model in best_models.items():
    print(f"  {name}: {type(model).__name__}")

## 7. Model Evaluation

In [ ]:
# Evaluate all models
evaluation_results = {}

for name, model in best_models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print('='*60)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.decision_function(X_test_scaled)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    evaluation_results[name] = {
        'metrics': metrics,
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    
    # Print metrics
    print(f"\nMetrics:")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1_score']:.4f}")
    print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    
    # Confusion Matrix
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    # Classification Report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

### 7.1 Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, results) in enumerate(evaluation_results.items()):
    y_pred = results['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_title(f'{name}')
    axes[idx].set_xticklabels(['No Disease', 'Disease'])
    axes[idx].set_yticklabels(['No Disease', 'Disease'])

plt.suptitle('Confusion Matrices Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

### 7.2 ROC Curves

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#3498db', '#e74c3c', '#2ecc71']

for idx, (name, results) in enumerate(evaluation_results.items()):
    y_proba = results['y_proba']
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    
    ax.plot(fpr, tpr, color=colors[idx], lw=2, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Model Comparison

In [ ]:
# Create comparison DataFrame
comparison_data = []

for name, results in evaluation_results.items():
    metrics = results['metrics']
    comparison_data.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1_score'],
        'ROC-AUC': metrics['roc_auc']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')

print("="*70)
print("MODEL COMPARISON TABLE")
print("="*70)
print(comparison_df.to_string(float_format='%.4f'))
print("="*70)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(comparison_df.index))
width = 0.15

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics):
    ax.bar(x + i * width, comparison_df[metric], width, label=metric, color=colors[i])

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(comparison_df.index)
ax.legend(loc='lower right')
ax.set_ylim([0, 1.1])
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/plots/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Best Model Selection

In [ ]:
# Find best model based on accuracy
best_model_name = comparison_df['Accuracy'].idxmax()
best_accuracy = comparison_df.loc[best_model_name, 'Accuracy']

print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {best_accuracy:.4f}")

best_model = best_models[best_model_name]

In [ ]:
# Save best model and scaler
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print(f"Best model saved to: models/best_model.pkl")
print(f"Scaler saved to: models/scaler.pkl")

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance = np.abs(best_model.coef_[0])
else:
    importance = None

if importance is not None:
    feature_importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance:")
    print(feature_importance_df.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(feature_importance_df['Feature'], feature_importance_df['Importance'],
                   color='#3498db', edgecolor='black')
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title('Feature Importance', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 10. Predictions

In [ ]:
# Sample patient data
sample_patient = {
    'age': 54,
    'sex': 1,  # Male
    'cp': 2,    # Non-anginal Pain
    'trestbps': 130,
    'chol': 250,
    'fbs': 0,
    'restecg': 1,
    'thalach': 150,
    'exang': 0,
    'oldpeak': 1.5,
    'slope': 2,
    'ca': 0,
    'thal': 2
}

print("Sample Patient Data:")
for key, value in sample_patient.items():
    print(f"  {key}: {value}")

In [ ]:
# Make prediction
features = np.array([[sample_patient[col] for col in feature_columns]])
features_scaled = scaler.transform(features)

prediction = best_model.predict(features_scaled)[0]
probability = best_model.predict_proba(features_scaled)[0][1]

print("\n" + "="*50)
print("PREDICTION RESULT")
print("="*50)
print(f"\nPrediction: {'High Risk of Heart Disease' if prediction == 1 else 'Low Risk of Heart Disease'}")
print(f"Risk Probability: {probability * 100:.2f}%")
print(f"Model Confidence: {max(probability, 1-probability) * 100:.2f}%")
print("="*50)

## 11. Conclusions

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)
print(f"\nDataset: Heart Disease Dataset ({len(df)} samples, {len(feature_columns)} features)")
print(f"\nBest Model: {best_model_name}")
print(f"\nModel Performance:")
print(f"  - Accuracy:  {comparison_df.loc[best_model_name, 'Accuracy']:.4f}")
print(f"  - Precision: {comparison_df.loc[best_model_name, 'Precision']:.4f}")
print(f"  - Recall:    {comparison_df.loc[best_model_name, 'Recall']:.4f}")
print(f"  - F1-Score:  {comparison_df.loc[best_model_name, 'F1-Score']:.4f}")
print(f"  - ROC-AUC:   {comparison_df.loc[best_model_name, 'ROC-AUC']:.4f}")
print(f"\nKey Insights:")
print(f"  - The model can predict heart disease with high accuracy")
print(f"  - Top predictive features: cp (chest pain), thalach (max heart rate), oldpeak (ST depression)")
print(f"  - The model is suitable for preliminary screening purposes")
print("\nNote: This model is for educational purposes and should not replace")
print("professional medical diagnosis.")
print("="*70)